# Colab notebook for fine-tuning

In [ ]:
!pip install transformers datasets pandas scikit-learn sentencepiece torch accelerate -q

In [ ]:
import pandas as pd
import logging
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
LOGGER.info(f"Using device: {device}")

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
TASK_PREFIX = "translate slang to standard: "

In [ ]:
def load_data(train_path, test_path):
    """Load train/test CSVs into Hugging Face Datasets."""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)
    
    LOGGER.info(f"Loaded {len(train_dataset)} training examples")
    LOGGER.info(f"Loaded {len(test_dataset)} test examples")
    
    return train_dataset, test_dataset

def preprocess_function(examples, tokenizer):
    """Tokenize inputs with task prefix and targets as labels."""
    inputs = [TASK_PREFIX + ex for ex in examples["source_text"]]
    targets = examples["target_text"]
    
    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    
    # Tokenize targets as labels
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
LOGGER.info(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
LOGGER.info(f"Model loaded. Parameters: {model.num_parameters():,.0f}")

In [ ]:
LOGGER.info("Tokenizing training dataset...")
train_dataset = train_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=["source_text", "target_text"],
)

LOGGER.info("Tokenizing test dataset...")
test_dataset = test_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=["source_text", "target_text"],
)

LOGGER.info(f"Train dataset columns: {train_dataset.column_names}")
LOGGER.info("Tokenization complete!")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="backend/fine_tuned_model",
    num_train_epochs=5,  # Adjust to 5-10 as needed
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    predict_with_generate=True,
    learning_rate=1e-4,
    seed=42,
    fp16=device == "cuda",  # Use fp16 if GPU available
)

LOGGER.info("Training configuration ready")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

LOGGER.info("Starting training...")
trainer.train()

In [ ]:
output_dir = "backend/fine_tuned_model"
LOGGER.info(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
LOGGER.info("Model and tokenizer saved!")
LOGGER.info("Training complete!")

In [ ]:
# Uncomment to share weights early via Google Drive
# from google.colab import files
# import shutil
# 
# # Zip the model checkpoint
# model_checkpoint = "backend/fine_tuned_model/checkpoint-500"  # Adjust checkpoint number
# shutil.make_archive("checkpoint", "zip", "backend/fine_tuned_model", "checkpoint-500")
# 
# # Download to local, then upload manually to Google Drive
# files.download("checkpoint.zip")
# LOGGER.info("Checkpoint zipped and ready for download to Google Drive")

In [ ]:
# Load the trained model for inference
trained_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
trained_tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Test example
test_input = "translate slang to standard: yo that's lowkey fire bro"
inputs = trained_tokenizer.encode(test_input, return_tensors="pt")
outputs = trained_model.generate(inputs, max_length=128)
result = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input: {test_input}")
print(f"Output: {result}")